# Цвиль Павел. Фит-231 Лабораторная работа №4

### Задание 1

Возьмите реализацию класса HashTable из лекционных материалов и выполните
следующие доработки:
1. Реализуйте квадратичное пробирование как технику повторного хеширования.
2. Реализуйте работу с функцией len (переопределите метод __len__).
3. Реализуйте работу оператора in (переопределите метод __contains__).
4. Переделайте метод put таким образом, чтобы таблица автоматически меняла размер,
когда загрузочный фактор становится больше значения 0.7. Размер должен
увеличиваться примерно в два раза до ближайшего подходящего простого числа.
5. Реализуйте работу оператора del (переопределите метод __delitem__) для удаления
элемента таблицы. Таблица должна автоматически менять размер, когда
загрузочный фактор становится меньше значения 0.2. Размер должен уменьшаться
примерно в два раза до ближайшего подходящего простого числа.
Все выполненные доработки должны быть протестированы.

In [6]:
class HashTable:
    def __init__(self, size=11):
        self.size = size                # размер таблицы (простое число)
        self.slots = [None] * size      # ключи
        self.data = [None] * size       # значения
        self.count = 0                  # количество элементов
    
    # Хеш-функция: метод остатков
    def hash(self, key):
        return key % self.size
    
    # Квадратичное пробирование: 
    def rehash(self, key, attempt):
        return (self.hash(key) + attempt**2) % self.size
    
    # Проверка, простое ли число
    def _is_prime(self, n):
        if n < 2:
            return False
        for i in range(2, int(n**0.5) + 1):
            if n % i == 0:
                return False
        return True
    
    # Следующее простое число
    def _next_prime(self, n):
        while not self._is_prime(n):
            n += 1
        return n
    
    # Изменить размер таблицы 
    def _resize(self, new_size):
        old_slots = self.slots[:]       # сохраняем старые ключи
        old_data = self.data[:]         # сохраняем старые значения
        self.size = self._next_prime(new_size)  # новый размер (простое число)
        self.slots = [None] * self.size
        self.data = [None] * self.size
        self.count = 0
        # Перехешируем все старые элементы
        for i in range(len(old_slots)):
            if old_slots[i] is not None:
                self.put(old_slots[i], old_data[i])
    
    # Добавить пару ключ-значение
    def put(self, key, val):
        # Проверяем загрузку и увеличиваем таблицу если нужно
        if self.count / self.size >= 0.7:
            self._resize(self.size * 2)
        
        attempt = 0
        h = self.hash(key)
        # Ищем свободный слот или такой же ключ
        while self.slots[h] is not None and self.slots[h] != key:
            attempt += 1
            h = self.rehash(key, attempt)
        
        if self.slots[h] is None:
            self.count += 1
        self.slots[h] = key
        self.data[h] = val
    
    # Получить значение по ключу
    def get(self, key):
        attempt = 0
        h = self.hash(key)
        start = h
        while self.slots[h] is not None:
            if self.slots[h] == key:
                return self.data[h]
            attempt += 1
            h = self.rehash(key, attempt)
            if h == start:      # вернулись в начало — ключа нет
                break
        return None
    
    def __len__(self):
        return self.count
    
    def __contains__(self, key):
        return self.get(key) is not None
    
    def __delitem__(self, key):
        attempt = 0
        h = self.hash(key)
        start = h
        while self.slots[h] is not None:
            if self.slots[h] == key:
                self.slots[h] = None
                self.data[h] = None
                self.count -= 1
                # Уменьшаем размер если загрузка < 0.2
                if self.count > 0 and self.count / self.size < 0.2 and self.size > 11:
                    self._resize(max(11, self.size // 2))
                return
            attempt += 1
            h = self.rehash(key, attempt)
            if h == start:
                break
        raise KeyError(key)
    
    def __str__(self):
        res = []
        for i in range(self.size):
            if self.slots[i] is not None:
                res.append(f"{self.slots[i]}: {self.data[i]}")
        return "{" + ", ".join(res) + "}"


ht = HashTable(11)

# Добавляем
for v in [54, 26, 93, 17, 77, 31, 44, 55, 20]:
    ht.put(v, f"val_{v}")

print("Таблица:", ht)
print("Размер:", len(ht))
print("77 в таблице?", 77 in ht)
print("100 в таблице?", 100 in ht)
print("ht.get(77) =", ht.get(77))

# Удаляем
del ht[77]
print("После удаления 77:", ht)
print("77 в таблице?", 77 in ht)

# Добавляем до автоувеличения
for i in range(20, 25):
    ht.put(i, str(i))
print("После добавления:", ht)
print("Итоговый размер таблицы:", ht.size)

Таблица: {93: val_93, 26: val_26, 77: val_77, 55: val_55, 54: val_54, 31: val_31, 17: val_17, 20: val_20, 44: val_44}
Размер: 9
77 в таблице? True
100 в таблице? False
ht.get(77) = val_77
После удаления 77: {93: val_93, 26: val_26, 55: val_55, 54: val_54, 31: val_31, 17: val_17, 20: val_20, 44: val_44}
77 в таблице? False
После добавления: {22: 22, 93: val_93, 24: 24, 26: val_26, 23: 23, 55: val_55, 54: val_54, 31: val_31, 17: val_17, 20: 20, 44: val_44, 21: 21}
Итоговый размер таблицы: 23


### Задание 2 

Возьмите реализацию класса HashTable из лекционных материалов и выполните
следующие доработки:
1. Переделайте существующие методы так, чтобы разрешение коллизий происходило
не при помощи концепции открытой адресации, а методом цепочек. Для этого в
каждом слоте храните связный список, реализованный классом UnorderedList из
лабораторной работы 3.
2. Реализуйте работу с функцией len (переопределите метод __len__).
3. Реализуйте работу оператора in (переопределите метод __contains__).
4. Переделайте метод put таким образом, чтобы таблица автоматически меняла размер,
когда загрузочный фактор становится больше значения 0.7. Размер должен
увеличиваться примерно в два раза до ближайшего подходящего простого числа.
5. Реализуйте работу оператора del (переопределите метод __delitem__) для удаления
элемента таблицы. Таблица должна автоматически менять размер, когда
загрузочный фактор становится меньше значения 0.2. Размер должен уменьшаться
примерно в два раза до ближайшего подходящего простого числа.
Все выполненные доработки должны быть протестированы.



In [8]:
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None

class UnorderedList:
    # Связный список из лабы 3
    def __init__(self):
        self.head = None
    
    def add(self, item):
        new = Node(item)
        new.next = self.head
        self.head = new
    
    def search(self, key):
        cur = self.head
        while cur:
            if cur.data[0] == key:
                return cur.data[1]
            cur = cur.next
        return None
    
    def update(self, key, val):
        cur = self.head
        while cur:
            if cur.data[0] == key:
                cur.data = (key, val)
                return True
            cur = cur.next
        return False

    def remove(self, key):
        cur = self.head
        prev = None
        while cur:
            if cur.data[0] == key:
                if prev:
                    prev.next = cur.next
                else:
                    self.head = cur.next
                return True
            prev = cur
            cur = cur.next
        return False
    
    def __str__(self):
        res = []
        cur = self.head
        while cur:
            res.append(str(cur.data))
            cur = cur.next
        return "[" + ", ".join(res) + "]"


class HashTableChained:
    def __init__(self, size=11):
        self.size = size                      # размер таблицы
        self.table = [UnorderedList() for _ in range(size)]  # в каждом слоте список
        self.count = 0                        # количество элементов
    
    def _hash(self, key):                     # хеш-функция — метод остатков
        return key % self.size
    
    def _is_prime(self, n):                   # проверка: простое ли число
        if n < 2:
            return False
        for i in range(2, int(n**0.5) + 1):
            if n % i == 0:
                return False
        return True
    
    def _next_prime(self, n):                 # следующее простое число
        while not self._is_prime(n):
            n += 1
        return n
    
    def _resize(self, new_size):              # изменить размер таблицы
        old_table = self.table
        self.size = self._next_prime(new_size)
        self.table = [UnorderedList() for _ in range(self.size)]
        self.count = 0
        for bucket in old_table:              # перехешируем все элементы
            cur = bucket.head
            while cur:
                self.put(cur.data[0], cur.data[1])
                cur = cur.next
    
    def put(self, key, val):                  # добавить пару ключ-значение
        if self.count / self.size > 0.7:      # загрузка > 0.7 -> увеличиваем
            self._resize(self.size * 2)
        
        h = self._hash(key)
        bucket = self.table[h]
        
        if bucket.update(key, val):           # ключ уже есть -> обновляем
            return
        bucket.add((key, val))                # новый ключ -> добавляем
        self.count += 1
    
    def get(self, key):                       # получить значение по ключу
        h = self._hash(key)
        return self.table[h].search(key)
    
    def __len__(self):                        # len()
        return self.count
    
    def __contains__(self, key):
        return self.get(key) is not None
    
    def __delitem__(self, key):
        h = self._hash(key)
        bucket = self.table[h]
        
        if bucket.remove(key):
            self.count -= 1
            if self.count > 0 and self.count / self.size < 0.2 and self.size > 11:
                self._resize(max(11, self.size // 2))  # загрузка < 20% -> уменьшаем
        else:
            raise KeyError(key)
    
    def __str__(self):
        res = []
        for i, bucket in enumerate(self.table):
            if bucket.head is not None:
                items = []
                cur = bucket.head
                while cur:
                    items.append(f"{cur.data[0]}: {cur.data[1]}")
                    cur = cur.next
                res.append(f"слот {i}: " + " -> ".join(items))
            else:
                res.append(f"слот {i}: пусто")
        return "\n".join(res)


ht = HashTableChained(7)
print("Размер таблицы: 7 (слоты 0-6)")

# Добавляем элементы
ht.put(10, "a")
ht.put(17, "b") #коллизия
ht.put(24, "c")
ht.put(3, "d")

print("\nТаблица после добавления:")
print(ht)

print(f"\nlen = {len(ht)}")
print(f"10 в таблице? {10 in ht}")
print(f"99 в таблице? {99 in ht}")
print(f"ht.get(17) = {ht.get(17)}")

# Удаляем элемент
del ht[10]
print("\nПосле удаления 10:")
print(ht)
print(f"10 в таблице? {10 in ht}")

Размер таблицы: 7 (слоты 0-6)

Таблица после добавления:
слот 0: пусто
слот 1: пусто
слот 2: пусто
слот 3: 3: d -> 24: c -> 17: b -> 10: a
слот 4: пусто
слот 5: пусто
слот 6: пусто

len = 4
10 в таблице? True
99 в таблице? False
ht.get(17) = b

После удаления 10:
слот 0: пусто
слот 1: пусто
слот 2: пусто
слот 3: 3: d -> 24: c -> 17: b
слот 4: пусто
слот 5: пусто
слот 6: пусто
10 в таблице? False


### Задание 3

Переделайте класс HashTable, чтобы в качестве ключей можно было использовать строки


In [9]:
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None

class UnorderedList:
    # Связный список из лабы 3
    def __init__(self):
        self.head = None
    
    def add(self, item):
        new = Node(item)
        new.next = self.head
        self.head = new
    
    def search(self, key):
        cur = self.head
        while cur:
            if cur.data[0] == key:
                return cur.data[1]
            cur = cur.next
        return None
    
    def update(self, key, val):
        cur = self.head
        while cur:
            if cur.data[0] == key:
                cur.data = (key, val)
                return True
            cur = cur.next
        return False
    
    def remove(self, key):
        cur = self.head
        prev = None
        while cur:
            if cur.data[0] == key:
                if prev:
                    prev.next = cur.next
                else:
                    self.head = cur.next
                return True
            prev = cur
            cur = cur.next
        return False
    
    def __str__(self):
        res = []
        cur = self.head
        while cur:
            res.append(str(cur.data))
            cur = cur.next
        return "[" + ", ".join(res) + "]"


class HashTableString:
    # Хеш-таблица для строковых ключей
    def __init__(self, size=7):
        self.size = size                      # размер таблицы
        self.table = [UnorderedList() for _ in range(size)]  # в каждом слоте список
        self.count = 0                        # количество элементов
    
    # Хеш-функция для строк: сумма кодов букв % размер
    def _hash(self, key):
        total = 0
        for ch in key:
            total += ord(ch)                  # ord() — код символа
        return total % self.size
    
    def _is_prime(self, n):                   # проверка: простое ли число
        if n < 2:
            return False
        for i in range(2, int(n**0.5) + 1):
            if n % i == 0:
                return False
        return True
    
    def _next_prime(self, n):                 # следующее простое число
        while not self._is_prime(n):
            n += 1
        return n
    
    def _resize(self, new_size):              # изменить размер таблицы
        old_table = self.table
        self.size = self._next_prime(new_size)
        self.table = [UnorderedList() for _ in range(self.size)]
        self.count = 0
        for bucket in old_table:              # перехешируем все элементы
            cur = bucket.head
            while cur:
                self.put(cur.data[0], cur.data[1])
                cur = cur.next
    
    def put(self, key, val):                  # добавить пару ключ-значение
        if self.count / self.size > 0.7:      # загрузка > 0.7 -> увеличиваем
            self._resize(self.size * 2)
        
        h = self._hash(key)
        bucket = self.table[h]
        
        if bucket.update(key, val):           # ключ уже есть -> обновляем
            return
        bucket.add((key, val))                # новый ключ -> добавляем
        self.count += 1
    
    def get(self, key):                       # получить значение по ключу
        h = self._hash(key)
        return self.table[h].search(key)
    
    def __len__(self):
        return self.count
    
    def __contains__(self, key):
        return self.get(key) is not None
    
    def __delitem__(self, key):
        h = self._hash(key)
        bucket = self.table[h]
        
        if bucket.remove(key):
            self.count -= 1
            if self.count > 0 and self.count / self.size < 0.2 and self.size > 7:
                self._resize(max(7, self.size // 2))
        else:
            raise KeyError(key)
    
    def __str__(self):
        res = []
        for i, bucket in enumerate(self.table):
            if bucket.head is not None:
                items = []
                cur = bucket.head
                while cur:
                    items.append(f"{cur.data[0]}: {cur.data[1]}")
                    cur = cur.next
                res.append(f"слот {i}: " + " -> ".join(items))
            else:
                res.append(f"слот {i}: пусто")
        return "\n".join(res)


ht = HashTableString(7)
print("Размер таблицы: 7 (слоты 0-6)")

# Добавляем строковые ключи
ht.put("кот", "мурчит")
ht.put("пёс", "гавкает")
ht.put("кот", "мяукает")      # обновляем значение
ht.put("мышь", "пищит")
ht.put("слон", "трубит")

print("\nТаблица после добавления:")
print(ht)

print(f"\nlen = {len(ht)}")
print(f"'кот' в таблице? {'кот' in ht}")
print(f"'корова' в таблице? {'корова' in ht}")
print(f"ht.get('кот') = {ht.get('кот')}")

# Удаляем элемент
del ht["кот"]
print("\nПосле удаления 'кот':")
print(ht)
print(f"'кот' в таблице? {'кот' in ht}")

Размер таблицы: 7 (слоты 0-6)

Таблица после добавления:
слот 0: пусто
слот 1: пусто
слот 2: пусто
слот 3: слон: трубит -> кот: мяукает
слот 4: мышь: пищит
слот 5: пёс: гавкает
слот 6: пусто

len = 4
'кот' в таблице? True
'корова' в таблице? False
ht.get('кот') = мяукает

После удаления 'кот':
слот 0: пусто
слот 1: пусто
слот 2: пусто
слот 3: слон: трубит
слот 4: мышь: пищит
слот 5: пёс: гавкает
слот 6: пусто
'кот' в таблице? False


### Задание 4

Дана строчка русского текста, состоящая из слов и пробелов. Словом считается
последовательность русских букв, слова разделены одним или большим числом пробелов.
Для каждого слова этого текста узнайте порядковый номер его вхождения в текст именно в
той форме, в которой указано слово. Для первого вхождения слова выведите «1», для
второго вхождения того же слова выведите «2» и так далее.
Для решения этой задачи используйте класс HashTable из задания № 3

In [12]:
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None

class UnorderedList:
    def __init__(self):
        self.head = None
    
    def add(self, item):
        new = Node(item)
        new.next = self.head
        self.head = new
    
    def search(self, key):
        cur = self.head
        while cur:
            if cur.data[0] == key:
                return cur.data[1]
            cur = cur.next
        return None
    
    def update(self, key, val):
        cur = self.head
        while cur:
            if cur.data[0] == key:
                cur.data = (key, val)
                return True
            cur = cur.next
        return False
    
    def __str__(self):
        res = []
        cur = self.head
        while cur:
            res.append(str(cur.data))
            cur = cur.next
        return "[" + ", ".join(res) + "]"


class HashTableString:
    def __init__(self, size=7):
        self.size = size
        self.table = [UnorderedList() for _ in range(size)]
        self.count = 0
    
    def _hash(self, key):
        total = 0
        for ch in key:
            total += ord(ch)
        return total % self.size
    
    def get(self, key):
        h = self._hash(key)
        return self.table[h].search(key)
    
    def put(self, key, value):
        h = self._hash(key)
        bucket = self.table[h]
        if bucket.update(key, value):
            return
        bucket.add((key, value))
        self.count += 1


text = input("Введите текст: ")
words = text.split()
print(text)

# Хеш-таблица для хранения счётчиков слов
ht = HashTableString(11)
result = []

for word in words:
    count = ht.get(word)          # сколько раз уже встречалось
    if count is None:
        ht.put(word, 1)           # первый раз
        result.append(1)
    else:
        ht.put(word, count + 1)   # увеличиваем счётчик
        result.append(count + 1)

print("\nРезультат:")
print(" ".join(map(str, result)))

кот собака кот кот собака мышь

Результат:
1 1 2 3 2 1


### Задание 5

Напишите программу, имитирующую процесс регистрации и авторизации. Для каждого
пользователя программа должна сохранять логин, хеш его пароля и «соль». Для хранения
данных можно использовать БД или файл.
Действия при сохранении пароля:
1. Сгенерируйте длинную случайную «соль» при помощи модуля secrets (secrets —
Generate secure random numbers for managing secrets — Python 3.10.0 documentation).
Длина «соли» должна быть такой же как и выходные данные используемой вами
хэш-функции. Например, если для хеширования вы используете SHA256, то на
выходе вы получите 256 бит (32 байта). В этом случае соль должна составлять не
менее 32 случайных байт.
2. Добавьте «соль» к паролю и хэшируйте его с помощью функции scrypt из модуля
hashlib (Хеширование паролей модулем hashlib в Python. (docs-python.ru)).
3. Сохраните логин, «соль» и получившейся хэш в БД или в файл. 

In [14]:
import secrets
import hashlib
import os

DB_FILE = "users.txt"

def hash_password(password, salt=None):
    # Хеширует пароль с солью
    if salt is None:
        salt = secrets.token_bytes(32)          # генерируем соль (32 байта)
    
    password_bytes = password.encode('utf-8')
    # scrypt — современная хеш-функция для паролей
    hash_obj = hashlib.scrypt(password_bytes, salt=salt, n=16384, r=8, p=1, dklen=32)
    
    return salt, hash_obj

def register():
    # Регистрация нового пользователя
    print("\nРЕГИСТРАЦИЯ")
    login = input("Логин: ")
    
    # Проверяем, существует ли пользователь
    if os.path.exists(DB_FILE):
        with open(DB_FILE, 'r') as f:
            for line in f:
                if line.startswith(login + ":"):
                    print("Ошибка: пользователь уже существует")
                    return
    
    password = input("Пароль: ")
    salt, hash_val = hash_password(password)
    
    # Сохраняем: логин:соль(hex):хеш(hex)
    with open(DB_FILE, 'a') as f:
        f.write(f"{login}:{salt.hex()}:{hash_val.hex()}\n")
    
    print("Регистрация успешна!")

def login():
    # Авторизация пользователя
    print("\nАВТОРИЗАЦИЯ")
    login = input("Логин: ")
    password = input("Пароль: ")
    
    if not os.path.exists(DB_FILE):
        print("Ошибка: нет зарегистрированных пользователей")
        return False
    
    with open(DB_FILE, 'r') as f:
        for line in f:
            parts = line.strip().split(':')
            if len(parts) == 3 and parts[0] == login:
                saved_login, salt_hex, hash_hex = parts
                salt = bytes.fromhex(salt_hex)
                saved_hash = bytes.fromhex(hash_hex)
                
                # Хешируем введённый пароль с той же солью
                _, test_hash = hash_password(password, salt)
                
                if test_hash == saved_hash:
                    print("Вход выполнен!")
                    return True
                else:
                    print("Неверный пароль")
                    return False
    
    print("Пользователь не найден")
    return False


while True:
    print("\n1 - Регистрация")
    print("2 - Вход")
    print("3 - Выход")
    
    choice = input("Выберите действие: ")
    
    if choice == "1":
        register()
    elif choice == "2":
        login()
    elif choice == "3":
        print("До свидания!")
        break
    else:
        print("Неверный выбор")


1 - Регистрация
2 - Вход
3 - Выход

АВТОРИЗАЦИЯ
Вход выполнен!

1 - Регистрация
2 - Вход
3 - Выход
До свидания!


### Задание 6

Напишите программу, которая принимает от пользователя путь до директории. Для всех
файлов из данной директории должен быть вычислен хеш. Программа должна выявить и
вывести на экран все дубликаты в этой директории (т.е. файлы, у которых одинаковый хеш).

In [19]:
import os
import hashlib

def get_hash(filename):
    with open(filename, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

path = input("Введите путь к папке: ")

files = os.listdir(path)              # список файлов в папке
hash_map = {}

for f in files:
    full_path = os.path.join(path, f)
    if os.path.isfile(full_path):      # только файлы, не папки
        h = get_hash(full_path)
        if h not in hash_map:
            hash_map[h] = []
        hash_map[h].append(f)

for h, names in hash_map.items():
    if len(names) > 1:
        print(f"\nОдинаковые файлы (хеш {h[:8]}...):")
    
        for name in names:
            print(f"  - {name}")
    else:
        print("Дубликатов нет")

Дубликатов нет
Дубликатов нет
Дубликатов нет
Дубликатов нет
Дубликатов нет
Дубликатов нет
Дубликатов нет
